# EDA

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
import re

import sys

sys.path.append("..")

In [85]:
DATA_INVOICES_CSV_PATH = "../data/invoices.csv"
DATA_PAYMENTS_CSV_PATH = "../data/payments.csv"

In [86]:
df_invoices = pd.read_csv(
    DATA_INVOICES_CSV_PATH, parse_dates=["invoice_date", "due_date"]
)
df_invoices.drop_duplicates(inplace=True)
df_invoices.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   invoice_id    8 non-null      str           
 1   vendor        8 non-null      str           
 2   invoice_date  8 non-null      datetime64[us]
 3   due_date      8 non-null      datetime64[us]
 4   currency      8 non-null      str           
 5   amount        8 non-null      float64       
 6   po_number     8 non-null      str           
 7   status        8 non-null      str           
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 644.0 bytes


In [87]:
df_payments = pd.read_csv(DATA_PAYMENTS_CSV_PATH, parse_dates=["payment_date"])
df_payments.drop_duplicates(inplace=True)
df_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   payment_id    10 non-null     str           
 1   payment_date  10 non-null     datetime64[us]
 2   payer_name    10 non-null     str           
 3   currency      10 non-null     str           
 4   amount        10 non-null     float64       
 5   reference     10 non-null     str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 612.0 bytes


In [88]:
pd.concat(
    [
        df_payments["reference"].str.extract(r"(INV-\d{4})"),
        df_payments["reference"].str.extract(r"(\b\d{4}\b)"),
        df_payments["reference"].str.extract(r"(PO-\d{4})"),
        df_payments["reference"].str.extract(r"(?i)\b(invoice)\b"),
        df_payments["reference"].str.extract(r"(?i)\b(INV\d{4})\b"),
    ],
    axis=1,
)

,0,0,0,0,0
0,NaN,1001,NaN,invoice,NaN
1,NaN,NaN,NaN,NaN,INV1002
2,NaN,8893,PO-8893,NaN,NaN
3,NaN,8894,PO-8894,NaN,NaN
4,NaN,1005,NaN,Invoice,NaN
5,NaN,8896,PO-8896,NaN,NaN
6,INV-1007,1007,NaN,NaN,NaN
7,INV-1007,1007,NaN,NaN,NaN
8,INV-1008,1008,NaN,NaN,NaN
9,NaN,NaN,NaN,invoice,NaN


In [89]:
# 1. Atrapa los directos: "INV-1001" y "INV1002" desde el inicio.
numeros_inv = df_payments["reference"].str.extract(r"(?i)\bINV-?(\d{4})(?!\d)")[0]
df_payments["EXTRACTED_invoice_id"] = "INV-" + numeros_inv

# 2. Atrapa los POs
df_payments["EXTRACTED_po_number"] = df_payments["reference"].str.extract(r"(PO-\d{4})")

# 3. Detecta la palabra "invoice"
df_payments["BOOL_CONTAINS_INVOICE_WORD"] = df_payments["reference"].str.contains(
    r"(?i)\b(invoice)\b", na=False, regex=True
)

# 4. Detecta exactamente 4 dígitos aislados de otros números
df_payments["BOOL_CONTAINS_FOUR_DIGITS"] = df_payments["reference"].str.contains(
    r"(?<!\d)\d{4}(?!\d)", na=False, regex=True
)

# 5. Rellena los huecos de EXTRACTED_invoice_id cuando dice "invoice 1003"
mask = df_payments.BOOL_CONTAINS_INVOICE_WORD & df_payments.BOOL_CONTAINS_FOUR_DIGITS
df_payments.loc[mask, "EXTRACTED_invoice_id"] = (
    "INV-" + df_payments.loc[mask, "reference"].str.extract(r"(?<!\d)(\d{4})(?!\d)")[0]
)

/tmp/ipykernel_51736/275178631.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_payments['BOOL_CONTAINS_INVOICE_WORD'] = df_payments['reference'].str.contains(


In [90]:
df_notes = pd.read_json("../data/notes.json")
df_notes

,source,text
0,email,ACME Logistics confirmed payment for invoice I...
1,email,Blue Ocean Supplies sent a partial payment for...
2,slack,Delta Tech Services applied a 10 USD early pay...
3,ops-note,Prime Retail Ops may have accidentally sent th...
4,email,Nova Packaging normally invoices in USD. Any E...


In [91]:
numeros_inv = df_notes["text"].str.extract(r"(?i)\bINV-?(\d{4})(?!\d)")[0]

df_notes["EXTRACTED_invoice_id"] = "INV-" + numeros_inv

df_notes["EXTRACTED_po_number"] = df_notes["text"].str.extract(r"(PO-\d{4})")

In [92]:
#  Matched, Partial Match, Needs Review, Unmatched, Suspicious
df_notes["text"].str.cat(sep=" ", na_rep="")

'ACME Logistics confirmed payment for invoice INV-1001. There was a typo in the payer name from the bank export. Blue Ocean Supplies sent a partial payment for PO-8893. Remaining balance will be paid next week. Delta Tech Services applied a 10 USD early payment discount for invoice INV-1005. Prime Retail Ops may have accidentally sent the same payment twice for invoice INV-1007. Please verify before closing. Nova Packaging normally invoices in USD. Any EUR payment should be manually reviewed before reconciliation.'

In [95]:
df_merged = (
    df_invoices.merge(
        df_payments,
        how="left",
        left_on="invoice_id",
        right_on="EXTRACTED_invoice_id",
        suffixes=("_invoice_tbl", "_payment_tbl"),
    )
    .drop(
        columns=[
            "EXTRACTED_invoice_id",
            "EXTRACTED_po_number",
            "BOOL_CONTAINS_INVOICE_WORD",
            "BOOL_CONTAINS_FOUR_DIGITS",
        ]
    )
    .merge(
        df_payments,
        how="left",
        left_on="po_number",
        right_on="EXTRACTED_po_number",
        suffixes=("_first_merge", "_second_payment_tbl"),
    )
    .drop(
        columns=[
            "EXTRACTED_invoice_id",
            "EXTRACTED_po_number",
            "BOOL_CONTAINS_INVOICE_WORD",
            "BOOL_CONTAINS_FOUR_DIGITS",
        ]
    )
)

df_merged["payment_id"] = df_merged["payment_id_first_merge"].fillna(
    df_merged["payment_id_second_payment_tbl"]
)
df_merged = df_merged.drop(
    columns=["payment_id_first_merge", "payment_id_second_payment_tbl"]
)

df_merged["payment_date"] = df_merged["payment_date_first_merge"].fillna(
    df_merged["payment_date_second_payment_tbl"]
)
df_merged = df_merged.drop(
    columns=["payment_date_first_merge", "payment_date_second_payment_tbl"]
)

df_merged["payer_name"] = df_merged["payer_name_first_merge"].fillna(
    df_merged["payer_name_second_payment_tbl"]
)
df_merged = df_merged.drop(
    columns=["payer_name_first_merge", "payer_name_second_payment_tbl"]
)

df_merged["reference"] = df_merged["reference_first_merge"].fillna(
    df_merged["reference_second_payment_tbl"]
)
df_merged = df_merged.drop(
    columns=["reference_first_merge", "reference_second_payment_tbl"]
)

df_merged["currency_payments"] = df_merged["currency_payment_tbl"].fillna(
    df_merged["currency"]
)
df_merged = df_merged.drop(columns=["currency_payment_tbl", "currency"])

df_merged["amount_payments"] = df_merged["amount_payment_tbl"].fillna(
    df_merged["amount"]
)
df_merged = df_merged.drop(columns=["amount_payment_tbl", "amount"])

df_merged["IS_DUE_PAYMENT"] = df_merged["payment_date"] > df_merged["due_date"]
df_merged.info()
# df_invoices
# df_payments

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   invoice_id            9 non-null      str           
 1   vendor                9 non-null      str           
 2   invoice_date          9 non-null      datetime64[us]
 3   due_date              9 non-null      datetime64[us]
 4   currency_invoice_tbl  9 non-null      str           
 5   amount_invoice_tbl    9 non-null      float64       
 6   po_number             9 non-null      str           
 7   status                9 non-null      str           
 8   payment_id            9 non-null      str           
 9   payment_date          9 non-null      datetime64[us]
 10  payer_name            9 non-null      str           
 11  reference             9 non-null      str           
 12  currency_payments     9 non-null      str           
 13  amount_payments       9 non-null   

In [96]:
df_merged

,invoice_id,vendor,invoice_date,due_date,currency_invoice_tbl,amount_invoice_tbl,po_number,status,payment_id,payment_date,payer_name,reference,currency_payments,amount_payments,IS_DUE_PAYMENT
0,INV-1001,ACME Logistics,2026-05-01,2026-05-30,USD,1250.0,PO-8891,open,PAY-9001,2026-05-15,ACME Logistcs,Payment for invoice 1001,USD,1250.0,False
1,INV-1002,Grupo Norte SA,2026-05-03,2026-06-02,USD,980.5,PO-8892,open,PAY-9002,2026-05-18,Grupo Norte,INV1002,USD,980.5,False
2,INV-1003,Blue Ocean Supplies,2026-05-04,2026-06-04,USD,4300.0,PO-8893,open,PAY-9003,2026-05-20,Blue Ocean Supplies,Partial payment PO-8893,USD,3000.0,False
3,INV-1004,ACME Logistics LLC,2026-05-08,2026-06-08,USD,700.0,PO-8894,open,PAY-9004,2026-05-22,Unknown Vendor,PO-8894,USD,700.0,False
4,INV-1005,Delta Tech Services,2026-05-10,2026-06-10,USD,1500.0,PO-8895,open,PAY-9005,2026-05-25,Delta Technology Services,Invoice 1005 discount applied,USD,1490.0,False
5,INV-1006,Northwind Foods,2026-05-11,2026-06-11,MXN,2200.0,PO-8896,open,PAY-9006,2026-05-26,Northwind Food,PO-8896,MXN,2200.0,False
6,INV-1007,Prime Retail Ops,2026-05-12,2026-06-12,USD,750.0,PO-8897,open,PAY-9007,2026-05-27,Prime Retail Ops,Payment for INV-1007,USD,750.0,False
7,INV-1007,Prime Retail Ops,2026-05-12,2026-06-12,USD,750.0,PO-8897,open,PAY-9008,2026-05-28,Prime Retail Operations,Second payment for INV-1007,USD,750.0,False
8,INV-1008,Nova Packaging,2026-05-14,2026-06-14,USD,1200.0,PO-8898,open,PAY-9009,2026-05-29,Nova Packaging,INV-1008,EUR,1200.0,False


In [115]:
df_final = (
    df_merged.merge(
        df_notes[["EXTRACTED_invoice_id", "source", "text"]],
        how="left",
        left_on="invoice_id",
        right_on="EXTRACTED_invoice_id",
    )
    .merge(
        df_notes[["EXTRACTED_po_number", "source", "text"]],
        how="left",
        left_on="po_number",
        right_on="EXTRACTED_po_number",
    )
    .drop(columns=["EXTRACTED_invoice_id", "EXTRACTED_po_number"])
)

df_final["source"] = df_final["source_x"].fillna(df_final["source_y"])
df_final = df_final.drop(columns=["source_x", "source_y"])

df_final["text"] = df_final["text_x"].fillna(df_final["text_y"])
df_final = df_final.drop(columns=["text_x", "text_y"])

# 'reference' + 'text' : flag
df_final["ALL_REFERENCES"] = df_final["reference"].fillna("") + df_final["text"].fillna(
    ""
)

df_final

,invoice_id,vendor,invoice_date,due_date,currency_invoice_tbl,amount_invoice_tbl,po_number,status,payment_id,payment_date,payer_name,reference,currency_payments,amount_payments,IS_DUE_PAYMENT,source,text,ALL_REFERENCES
0,INV-1001,ACME Logistics,2026-05-01,2026-05-30,USD,1250.0,PO-8891,open,PAY-9001,2026-05-15,ACME Logistcs,Payment for invoice 1001,USD,1250.0,False,email,ACME Logistics confirmed payment for invoice I...,Payment for invoice 1001ACME Logistics confirm...
1,INV-1002,Grupo Norte SA,2026-05-03,2026-06-02,USD,980.5,PO-8892,open,PAY-9002,2026-05-18,Grupo Norte,INV1002,USD,980.5,False,NaN,NaN,INV1002
2,INV-1003,Blue Ocean Supplies,2026-05-04,2026-06-04,USD,4300.0,PO-8893,open,PAY-9003,2026-05-20,Blue Ocean Supplies,Partial payment PO-8893,USD,3000.0,False,email,Blue Ocean Supplies sent a partial payment for...,Partial payment PO-8893Blue Ocean Supplies sen...
3,INV-1004,ACME Logistics LLC,2026-05-08,2026-06-08,USD,700.0,PO-8894,open,PAY-9004,2026-05-22,Unknown Vendor,PO-8894,USD,700.0,False,NaN,NaN,PO-8894
4,INV-1005,Delta Tech Services,2026-05-10,2026-06-10,USD,1500.0,PO-8895,open,PAY-9005,2026-05-25,Delta Technology Services,Invoice 1005 discount applied,USD,1490.0,False,slack,Delta Tech Services applied a 10 USD early pay...,Invoice 1005 discount appliedDelta Tech Servic...
5,INV-1006,Northwind Foods,2026-05-11,2026-06-11,MXN,2200.0,PO-8896,open,PAY-9006,2026-05-26,Northwind Food,PO-8896,MXN,2200.0,False,NaN,NaN,PO-8896
6,INV-1007,Prime Retail Ops,2026-05-12,2026-06-12,USD,750.0,PO-8897,open,PAY-9007,2026-05-27,Prime Retail Ops,Payment for INV-1007,USD,750.0,False,ops-note,Prime Retail Ops may have accidentally sent th...,Payment for INV-1007Prime Retail Ops may have ...
7,INV-1007,Prime Retail Ops,2026-05-12,2026-06-12,USD,750.0,PO-8897,open,PAY-9008,2026-05-28,Prime Retail Operations,Second payment for INV-1007,USD,750.0,False,ops-note,Prime Retail Ops may have accidentally sent th...,Second payment for INV-1007Prime Retail Ops ma...
8,INV-1008,Nova Packaging,2026-05-14,2026-06-14,USD,1200.0,PO-8898,open,PAY-9009,2026-05-29,Nova Packaging,INV-1008,EUR,1200.0,False,NaN,NaN,INV-1008


In [ ]:
# IMPORTANTE: El orden de las reglas importa. La primera que coincida se aplica.
RULES = [
    ("Suspicious", r"twice|duplicate|same payment|accidentally|doble|double"),
    (
        "Needs Review",
        r"verify|manually review|review before|EUR.*payment|any.*EUR|mismatch",
    ),
    (
        "Partial Match",
        r"partial payment|partial|remaining balance|balance will be paid|installment",
    ),
    (
        "Matched",
        r"confirmed payment|confirmed|applied.*discount|payment applied|invoice.*paid",
    ),
]


def classify_row(text: str) -> str:
    for flag, pattern in RULES:
        if re.search(pattern, text, re.IGNORECASE):
            return flag
    return "Unmatched"


# df_notes["flag"] = df_notes["text"].apply(classify_row)

In [ ]:
# Layer 1: text-based classification
df_final["flag"] = df_final["ALL_REFERENCES"].apply(classify_row)

# Layer 2: data signals for rows that text couldn't classify
def layer2(row):
    if row["flag"] != "Unmatched":
        return row["flag"]
    if row["IS_DUE_PAYMENT"]:
        return "Needs Review"
    if row["currency_invoice_tbl"] != row["currency_payments"]:
        return "Needs Review"
    if row["payer_name"] == "Unknown Vendor":
        return "Needs Review"
    if row["amount_invoice_tbl"] != row["amount_payments"]:
        return "Partial Match" if row["amount_payments"] < row["amount_invoice_tbl"] else "Needs Review"
    if pd.notna(row["payment_id"]):
        return "Matched"
    return "Unmatched"

df_final["flag"] = df_final.apply(layer2, axis=1)
df_final[["invoice_id", "payer_name", "flag", "ALL_REFERENCES"]]